In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
h2 = ['gdp', 'inflation_df', 'agr_land', 'livestock_prod']

### Import the data
* Import data that describe countries in one point

In [3]:
one_point_countries_win = pd.read_parquet('../data/one_point_countries_winsorized.parquet')

In [4]:
one_point_countries_win.head()

,Country,labor_force_mean,labor_force_median,labor_force_std,labor_force_min,labor_force_max,labor_force_last,livestock_prod_mean,livestock_prod_median,livestock_prod_std,...,govermment_effective__mean,govermment_effective__median,govermment_effective__cv,govermment_effective__min,govermment_effective__max,govermment_effective__last,govermment_effective__growth,govermment_effective__skew,govermment_effective__kurtosis,govermment_effective__trend
0,Australia,2.679788,2.679994,0.481437,2.619583,2.740840,2.740840,4.582413,4.585987,1.820075,...,0.907532,0.896973,0.138675,0.852072,0.783123,0.991862,-0.183282,-0.557676,0.418830,0.517477
1,Austria,1.736775,1.731656,0.088532,1.717395,1.763017,1.763017,4.614130,4.615121,1.027869,...,0.800186,0.807935,0.138675,0.812366,0.810674,0.651612,-0.219351,-0.255483,0.418830,-0.060887
2,Belgium,1.825355,1.821318,0.140643,1.791759,1.859418,1.859418,4.636378,4.628887,1.027869,...,0.423795,0.390140,0.138675,0.428533,0.521389,0.272102,-0.262785,0.825197,-0.006636,-0.915149
3,Brazil,3.719821,3.718924,0.695575,3.706474,3.736955,3.733375,4.672829,4.675629,1.646041,...,-1.172808,-1.123510,0.138675,-1.000910,-1.200545,-1.023467,-0.126735,0.825197,-0.714821,-0.915149
4,Bulgaria,1.448095,1.451614,0.091234,1.408545,1.477049,1.408545,4.582413,4.585987,1.688591,...,-1.172808,-1.109812,0.138675,-1.000910,-1.172995,-1.023467,-0.268474,-0.822060,0.057350,-0.412401


In [5]:
one_point_countries_win.shape

(53, 189)

### Delete macro statistics

In [6]:
def filter_macro_columns_with_country(df, macro_keywords, keep_column="Country"):
    """
    Conserva solo las columnas del DataFrame que contienen alguna palabra clave en 'macro_keywords',
    además de la columna 'Country'.

    Parámetros:
    - df: pandas.DataFrame
        El DataFrame original con muchas columnas macroeconómicas.
    - macro_keywords: list of str
        Lista de palabras clave que deben estar presentes en el nombre de las columnas a conservar.
    - keep_column: str
        Columna adicional que se desea conservar (por defecto 'Country').

    Retorna:
    - df_filtrado: pandas.DataFrame
        DataFrame con solo las columnas relevantes.
    """

    columnas_filtradas = [
        col for col in df.columns
        if any(keyword in col for keyword in macro_keywords)
    ]

    # Asegurar que 'Country' esté incluido si existe en el DataFrame
    if keep_column in df.columns and keep_column not in columnas_filtradas:
        columnas_filtradas.insert(0, keep_column)

    df_filtrado = df[columnas_filtradas].copy()

    print(f"Columnas conservadas ({len(columnas_filtradas)}): {columnas_filtradas}")
    return df_filtrado

In [7]:
countries_win6_h2 = filter_macro_columns_with_country(one_point_countries_win, h2)

Columnas conservadas (34): ['Country', 'livestock_prod_mean', 'livestock_prod_median', 'livestock_prod_std', 'livestock_prod_cv', 'livestock_prod_min', 'livestock_prod_max', 'livestock_prod_last', 'gdp_mean', 'gdp_median', 'gdp_std', 'gdp_cv', 'gdp_min', 'gdp_max', 'gdp_last', 'gdp_growth', 'gdp_trend', 'livestock_prod_growth', 'livestock_prod_skew', 'livestock_prod_kurtosis', 'livestock_prod_trend', 'agr_land_mean', 'agr_land_median', 'agr_land_std', 'agr_land_cv', 'agr_land_min', 'agr_land_max', 'agr_land_last', 'agr_land_growth', 'agr_land_skew', 'agr_land_kurtosis', 'agr_land_trend', 'gdp_skew', 'gdp_kurtosis']


In [8]:
countries_win6_h2.shape

(53, 34)

In [9]:
def detect_outliers_in_dataframe(df, iqr_multiplier=1.5):
    outlier_report = {}

    # Excluye columnas no numéricas
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for idx, row in df.iterrows():
        country_name = row["Country"]
        outlier_cols = []

        for col in numeric_cols:
            value = row[col]
            col_series = df[col].dropna()

            # Calcula los límites usando IQR
            q1 = np.percentile(col_series, 25)
            q3 = np.percentile(col_series, 75)
            iqr = q3 - q1
            lower_bound = q1 - iqr_multiplier * iqr
            upper_bound = q3 + iqr_multiplier * iqr

            if value < lower_bound or value > upper_bound:
                outlier_cols.append(col)

        outlier_report[country_name] = outlier_cols

    return outlier_report

In [10]:
detect_outliers_in_dataframe(countries_win6_h2)

{'Australia': [],
 'Austria': [],
 'Belgium': [],
 'Brazil': [],
 'Bulgaria': [],
 'Canada': [],
 'Chile': [],
 'Colombia': [],
 'Croatia': [],
 'Cyprus': [],
 'Czech': [],
 'Denmark': [],
 'Estonia': [],
 'Finland': [],
 'France': [],
 'Germany': [],
 'Greece': [],
 'Hungary': [],
 'Iceland': [],
 'India': [],
 'Ireland': [],
 'Israel': [],
 'Italy': [],
 'Japan': [],
 'Latvia': [],
 'Lithuania': [],
 'Luxembourg': [],
 'Malaysia': [],
 'Malta': [],
 'Mexico': [],
 'Netherlands': [],
 'New Zealand': [],
 'Norway': [],
 'Paraguay': [],
 'Peru': [],
 'Philippines': [],
 'Poland': [],
 'Portugal': [],
 'Romania': [],
 'Russia': [],
 'Serbia': [],
 'Singapore': [],
 'Slovakia': [],
 'Slovenia': [],
 'South Africa': [],
 'South Korea': [],
 'Spain': [],
 'Sweden': [],
 'Switzerland': [],
 'Thailand': [],
 'Turkey': [],
 'United Kingdom': [],
 'USA': []}